## Lib integration

In [1]:
import sys
# sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching")
sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching\demo-code\2d")
import h5py
from collections import defaultdict, Counter
import numpy as np
from rich import print

In [2]:
file_path = r"E:\Dai hoc\2526I\dacn\flow-matching\data\traintest_hcd.hdf5"

## Open data

In [3]:
with h5py.File(file_path, "r") as f:
    print("Keys:", list(f.keys()))

    seqs = f["sequence_integer"][:10024]
    charges_oh = f["precursor_charge_onehot"][:10024]
    intensities = f["intensities_raw"][:10024]  

Keys:
[
    'collision_energy',
    'collision_energy_aligned',
    'collision_energy_aligned_normed',
    'intensities_raw',
    'masses_pred',
    'masses_raw',
    'method',
    'precursor_charge_onehot',
    'rawfile',
    'reverse',
    'scan_number',
    'score',
    'sequence_integer',
    'sequence_onehot'
]

In [4]:
charges = np.argmax(charges_oh, axis=1) + 1
del charges_oh

### Utils

### Peak data

In [5]:
# np.argmax(charges_oh, axis=1) + 1

In [6]:
len(intensities[0])

174

In [7]:
# max_index = 0
# for seq in seqs:
#     for token in seq:
#         if token > max_index:
#             max_index = token
# print("Max token index:", max_index)

## Explore data

## FLow matching training

In [8]:
import torch
torch.set_default_device("cuda")
from torch import nn
# from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import imageio
from sklearn.datasets import make_moons
import math

from coupling import mini_batch_coupling, greedy_coupling, sinkhorm_coupling
from draw import DrawFlow, DrawSample
from gen_path import get_xt
from flow_model import HCDFlowResMLP, HCDFlow

### Training configuration

In [9]:
epoch = 1e6
batch_size = 1024
model = HCDFlow(noise_dim=174, pep_dim=256, time_dim=128, charge_dim=128)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, eps=1e-10)

#### Tranable params

In [10]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

5615790

In [11]:
# from tqdm.auto import tqdm
# # pbar = tqdm(range(int(epoch)), desc="Training")
# loss_history = []
# pbar = tqdm(range(int(epoch)), desc="Training")
# for ep in pbar:
#     model.train()
#     batch_indices = np.random.choice(len(intensities), batch_size, replace=False)
#     batch_intensities = torch.tensor(intensities[batch_indices], dtype=torch.float32)
#     batch_pep_seq = torch.tensor(seqs[batch_indices], dtype=torch.long)
#     batch_charge = torch.tensor(charges[batch_indices], dtype=torch.float32).unsqueeze(1)
    
#     noise = torch.randn_like(batch_intensities)
#     t = torch.rand(batch_size, 1)
#     x_t = get_xt(batch_intensities, noise, t)
#     # print(x_t.shape, t.shape, batch_pep_seq.shape, batch_charge.shape)
#     u_pred = model(x_t, t=t, pep_seq=batch_pep_seq, charge=batch_charge)
#     loss = nn.MSELoss()(u_pred, batch_intensities - noise)
#     optimizer.zero_grad()

#     loss.backward()

#     optimizer.step()

#     loss_history.append(loss.item())
#     if (ep + 1) % 100 == 0:
#        pbar.set_postfix({"Loss": f"{loss.item():.4f}", "Avg Loss 100": f"{(sum(loss_history)/len(loss_history)):.4f}"})

## Pearson Correlation Coefficent

In [12]:
# pcc cal
def pcc(Y_true: torch.Tensor, Y_pred: torch.Tensor):
    # Y_true and Y_pred are arrays of intensities
    Y_true_mean = Y_true.mean(dim=0, keepdim=True)
    Y_pred_mean = Y_pred.mean(dim=0, keepdim=True)
    Y_true_centered = Y_true - Y_true_mean
    Y_pred_centered = Y_pred - Y_pred_mean
    covariance = (Y_true_centered * Y_pred_centered).sum(dim=0)
    Y_true_std = Y_true_centered.pow(2).sum(dim=0).sqrt()
    Y_pred_std = Y_pred_centered.pow(2).sum(dim=0).sqrt()
    pcc = covariance / (Y_true_std * Y_pred_std + 1e-8)
    return pcc.mean().item()

In [13]:
test_seq = seqs[0]
test_charge = charges[0]
test_intensities = []
for seq, charge, intensity in zip(seqs, charges, intensities):
    
    if np.array_equal(seq, test_seq) and test_charge == charge:
        test_intensities.append(intensity)
len(test_intensities)

1

In [17]:
with torch.no_grad():
    model.eval()
    test_pep_seq = torch.tensor(test_seq, dtype=torch.long).unsqueeze(0)
    test_charge_tensor = torch.tensor(test_charge, dtype=torch.float32).unsqueeze(0).unsqueeze(1)
    test_intensities_tensor = torch.tensor(test_intensities, dtype=torch.float32)
    
    num_samples = 1
    noise = torch.randn(num_samples, *test_intensities_tensor[0].shape)
    
    test_pep_seq = test_pep_seq.repeat(num_samples, 1)
    test_charge_tensor = test_charge_tensor.repeat(num_samples, 1)
    pred_intentities = model.sample(noise,test_pep_seq, test_charge_tensor, step=10)
    print(pred_intentities - test_intensities_tensor)
    
    pcc_value = pcc(test_intensities_tensor, pred_intentities)
    print(f"PCC: {pcc_value:.4f}")

tensor([[-1.2972e+00,  4.4637e-01,  8.2613e-01,  2.4729e+00,  8.8112e-02,
          6.0897e-01,  8.7979e-01, -1.3519e+00,  1.2669e+00, -1.9102e-03,
          9.1339e-01,  7.6708e-01,  9.7121e-01, -6.6368e-01, -5.4447e-01,
          6.0390e-01, -4.0940e-01, -7.3251e-03,  7.6888e-02,  2.0664e+00,
         -1.0067e+00,  2.9763e-01, -1.0960e+00,  7.1465e-01, -7.0489e-01,
          7.0059e-01,  4.5402e-01, -1.0875e+00,  3.3965e-01,  6.5038e-01,
          4.3347e-01, -5.7126e-01,  7.7322e-01, -1.2498e+00, -1.1944e+00,
         -5.2890e-01, -1.2697e+00,  6.1967e-01,  4.1046e-01, -6.5560e-02,
         -9.8383e-01,  1.3220e+00,  4.5405e-01, -1.2565e+00, -9.7344e-01,
          1.4453e+00, -5.5189e-02,  2.5295e-01,  3.3151e-01,  7.5103e-02,
          3.2068e-01, -2.0532e+00,  1.3164e-01, -1.6958e-01, -1.8224e+00,
          1.2818e+00, -9.0596e-01, -6.9041e-02, -8.9289e-01,  6.6237e-02,
          1.8223e-01, -1.3990e+00, -2.5268e-02, -9.2035e-01, -3.5164e-01,
         -7.6363e-01,  6.6193e-01,  3.1173e-01,  2.2110e+00,  6.1514e-01,
         -2.4571e-01,  6.0606e-01,  1.1339e+00,  7.8081e-01, -2.5021e+00,
         -1.7216e-01, -7.6494e-01, -1.4100e+00, -1.8199e+00,  7.9663e-01,
          5.1872e-01, -1.9071e-01, -1.0401e+00,  1.5200e-02, -5.5723e-01,
         -1.0199e+00, -1.3576e+00, -3.8521e-01,  2.1690e-01, -1.6700e+00,
         -1.2533e+00, -1.0467e+00,  1.6480e+00, -7.1107e-02, -7.5754e-01,
         -7.4241e-01, -1.8144e+00, -6.0729e-01, -8.8122e-01, -5.0599e-01,
         -2.9535e-01, -3.1886e-01,  7.2416e-01,  8.7967e-02,  5.2286e-01,
          1.2130e+00,  1.6874e+00,  3.2391e-01,  1.1739e+00,  1.3232e+00,
         -2.5467e-01,  2.2738e+00,  2.8282e+00, -1.2244e-01,  7.7548e-01,
          2.3506e+00,  1.2356e-01,  1.4548e+00,  1.8772e+00,  5.7300e-01,
          1.4579e+00,  2.8385e+00, -1.0492e+00, -8.0383e-01,  4.8268e-01,
          3.6011e-01,  1.3307e+00, -1.3999e-01,  5.3009e-01,  4.3837e-01,
          1.3941e+00,  9.5757e-01,  9.9631e-01,  2.9115e+00,  2.7170e+00,
         -4.2704e-02,  2.2162e+00,  2.5398e+00,  1.2138e+00,  1.0253e+00,
         -9.6679e-04,  1.6759e+00,  1.2420e+00,  1.0750e+00, -3.0965e-01,
          4.7675e-03,  1.9912e+00,  9.8116e-01,  2.9470e-01,  3.5742e-01,
          1.8096e+00, -4.4392e-01,  3.0705e-01,  9.1620e-01,  2.2315e+00,
          1.4660e+00,  2.7137e+00,  7.2370e-01,  5.3098e-01,  1.0986e-01,
          3.2526e-01,  1.6623e+00,  2.9768e+00,  2.5676e+00,  1.7842e+00,
          5.1945e-01,  2.5423e+00,  2.0627e+00,  7.9124e-02,  2.0359e+00,
          1.2986e+00, -4.6890e-02,  1.3296e+00,  5.3781e-01]], device='cuda:0')

PCC: 0.0000

In [19]:
test_intensities_tensor.shape

torch.Size([1, 174])